# C7-cnn-transfer — Practice p11 — Solution

The hand counts separate the inherited prefix from the fresh head before any audit API is used.

In [ ]:
# Cache pin (course convention, plan 009): pretrained weights live in the repo's
# gitignored reference/cache/ -- resolve it from the repo root BEFORE importing torch.
import os, pathlib
_env_root = os.environ.get("USAAIO_BOOK_ROOT")
if _env_root:
    _root = pathlib.Path(_env_root).resolve()
else:
    _start = pathlib.Path.cwd().resolve()
    _root = next(
        p for p in [_start, *_start.parents]
        if (p / "syllabus.md").is_file() and (p / "curriculum").is_dir()
    )
os.environ["TORCH_HOME"] = str(_root / "reference" / "cache" / "torch")

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# float32 register (course exception): pretrained resnet50 is a float32 artifact.
# No float64 default here; inputs are cast .to(torch.float32) at the model
# boundary; repeat float32 forwards are bit-identical.
SEED = 20260804

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval()
assert next(model.parameters()).dtype == torch.float32

torch.manual_seed(SEED)
x = torch.randn(2, 3, 224, 224).to(torch.float32)


In [ ]:
# Hand arithmetic only: 9,408 + 128 + 215,808 + 1,219,584.
hand_frozen = 1_444_928
# A 512-to-9 affine head: 512 * 9 + 9.
hand_trainable = 4_617


In [ ]:
class Transfer9(nn.Module):
    def __init__(self, pretrained):
        super().__init__()
        self.backbone = nn.Sequential(*list(pretrained.children())[:6])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(512, 9)
        for p in self.backbone.parameters():
            p.requires_grad = False

    def forward(self, x):
        features = self.pool(self.backbone(x))
        return self.head(torch.flatten(features, 1))

net = Transfer9(model)
net.eval()
with torch.inference_mode():
    out_shape = tuple(net(x).shape)
n_frozen = sum(p.numel() for p in net.parameters() if not p.requires_grad)
n_trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
counts_match = n_frozen == hand_frozen and n_trainable == hand_trainable
trainable_names = sorted(name for name, p in net.named_parameters() if p.requires_grad)


### Answer check

In [ ]:
assert out_shape == (2, 9)
assert counts_match
assert (n_frozen, n_trainable) == (1_444_928, 4_617)
assert trainable_names == ["head.bias", "head.weight"]
